In [1]:
# Manmeet Singh
# Compare the performance of BFS and A* with 3x3 and 4x4 sliding puzzles.
# Generate random puzzle and analyze algorithm efficiency.

# library to use
import random
from collections import deque
import heapq
import pandas as pd

# Puzzle Functions
def valid_moves(state, size):
    """
    Return all possible moves (as index shifts) for the blank (0) in the given state.
    """
    idx = state.index(0)
    row, col = divmod(idx, size)
    moves = []
    if row > 0:
      moves.append(-size)     # Move Up
    if row < size - 1:
      moves.append(size)  # Move Down
    if col > 0:
      moves.append(-1)        # Move Left
    if col < size - 1:
      moves.append(1)  # Move Right
    return moves

def move(state, direction, size):
    """
    Apply a move to the puzzle state and return the new state.
    """
    idx = state.index(0)
    new_idx = idx + direction

    new_state = list(state)
    # swap
    new_state[idx], new_state[new_idx] = new_state[new_idx], new_state[idx]
    return new_state

def random_walk(start_state, steps, size):
    """
    Generate a random puzzle state by performing `steps` random valid moves from the goal.
    """
    current = start_state[:]
    for _ in range(steps):
        options = valid_moves(current, size)
        move_choice = random.choice(options)
        current = move(current, move_choice, size)
    return current

# BFS function
def bfs(start_state, goal_state, size):
    """
    Breadth-First Search (uninformed) to solve the puzzle.
    """
    frontier = deque([[start_state]])
    explored = set()
    nodes_expanded = 0

    while frontier:
        path = frontier.popleft()
        state = path[-1]
        if state == goal_state:
            return path, nodes_expanded
        explored.add(tuple(state))
        for action in valid_moves(state, size):
            child = move(state, action, size)
            if tuple(child) not in explored:
                frontier.append(path + [child])
        nodes_expanded += 1
    return None, nodes_expanded

# A* Search

def out_of_place(state, goal):
    # Count of tiles not in the correct position.
    return sum(1 for i in range(len(state)) if state[i] != 0 and state[i] != goal[i])

def manhattan(state, goal, size):
    # Sum of Manhattan distances for each tile from its goal position.
    distance = 0
    for num in range(1, size * size):
        i, j = state.index(num), goal.index(num)
        xi, yi = divmod(i, size)
        xj, yj = divmod(j, size)
        # sum the distance
        distance += abs(xi - xj) + abs(yi - yj)
    return distance

# Informed Search A*
def astar(start_state, goal_state, heuristic_func, size, max_nodes=100000):
    # A* Search using the given heuristic.
    frontier = []
    heapq.heappush(frontier, (0, [start_state]))
    explored = set()
    nodes_expanded = 0

    while frontier:
        cost, path = heapq.heappop(frontier)
        state = path[-1]
        nodes_expanded += 1
        if state == goal_state:
            return path, nodes_expanded
        if nodes_expanded > max_nodes:
            return None, nodes_expanded
        explored.add(tuple(state))
        for action in valid_moves(state, size):
            child = move(state, action, size)
            if tuple(child) not in explored:
                new_cost = len(path) + heuristic_func(child)
                heapq.heappush(frontier, (new_cost, path + [child]))
    return None, nodes_expanded


# Problem Generation using random goal step and size
def generate_problems(goal, size):
    # Generate 15 puzzles via random walk: 3 each from 5, 10, 20, 40, 80 steps.
    problems = []
    for steps in [5, 10, 20, 40, 80]:
        for _ in range(3):
            state = random_walk(goal, steps, size)
            problems.append((state, steps))
    return problems

# goal states for both puzzles
goal_3x3 = [1,2,3,4,5,6,7,8,0]
# a list from 1 to 15 and 0 in the end
goal_4x4 = list(range(1, 16)) + [0]

# Generate puzzles
problems_3x3 = generate_problems(goal_3x3, 3)
problems_4x4 = generate_problems(goal_4x4, 4)

# solving puzzles and Storing Results
results = []
def solve_all(problems, size, goal, name):
    """
    Run BFS, A* (Out-of-Place), and A* (Manhattan) on the generated puzzles.
    Record performance.
    """
    for i, (start, steps) in enumerate(problems, 1):
        if size == 3 or steps <= 10:
            path, nodes = bfs(start, goal, size)
            results.append({'Puzzle': name, 'Steps': steps, 'Algorithm': 'BFS', 'Solution Length': len(path)-1 if path else None, 'Nodes Expanded': nodes})

        path, nodes = astar(start, goal, lambda s, g=goal: out_of_place(s, g), size)
        results.append({'Puzzle': name, 'Steps': steps, 'Algorithm': 'A* Out-of-Place', 'Solution Length': len(path)-1 if path else None, 'Nodes Expanded': nodes})

        path, nodes = astar(start, goal, lambda s, g=goal: manhattan(s, g, size), size)
        results.append({'Puzzle': name, 'Steps': steps, 'Algorithm': 'A* Manhattan', 'Solution Length': len(path)-1 if path else None, 'Nodes Expanded': nodes})

# Run all evaluations
solve_all(problems_3x3, 3, goal_3x3, '3x3')
solve_all(problems_4x4, 4, goal_4x4, '4x4')

# Results
# Load results into DataFrame and group by algorithm performance
df = pd.DataFrame(results)
df_grouped = df.groupby(['Puzzle', 'Steps', 'Algorithm']).mean(numeric_only=True)
print(df_grouped)


                              Solution Length  Nodes Expanded
Puzzle Steps Algorithm                                       
3x3    5     A* Manhattan            3.000000        4.000000
             A* Out-of-Place         3.000000        4.000000
             BFS                     3.000000       15.000000
       10    A* Manhattan            4.000000        5.000000
             A* Out-of-Place         4.000000        5.000000
             BFS                     4.000000       25.666667
       20    A* Manhattan           10.666667       13.333333
             A* Out-of-Place        10.666667       38.333333
             BFS                    10.666667      855.333333
       40    A* Manhattan           11.333333      189.666667
             A* Out-of-Place        11.333333     1109.000000
             BFS                    11.333333    19965.000000
       80    A* Manhattan           14.000000       56.666667
             A* Out-of-Place        14.000000      213.666667
        